In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"




In [4]:
# ============================================================
# STEP 1 — Recompute I_parent independently (no leakage), save to catalog
# ============================================================
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

def compute_blendedness_with_iparent(df, ra_col="ra", dec_col="dec",
                                       flux_col="flux_r", sigma_px=6.0,
                                       trunc_px=30.0, pixel_scale=0.2):
    ra = df[ra_col].values
    dec = df[dec_col].values
    flux = df[flux_col].values
    n = len(df)
    dec_mean = np.deg2rad(dec.mean())
    x = (ra - ra.mean()) * np.cos(dec_mean) * 3600.0 / pixel_scale
    y = (dec - dec.mean()) * 3600.0 / pixel_scale
    coords = np.column_stack([x, y])
    tree = cKDTree(coords)
    pairs = tree.query_pairs(r=trunc_px, output_type="ndarray")
    i_idx, j_idx = pairs[:, 0], pairs[:, 1]
    d = np.linalg.norm(coords[i_idx] - coords[j_idx], axis=1)
    w = np.exp(-0.5 * (d / sigma_px) ** 2)
    neighbor_flux_sum = np.zeros(n)
    np.add.at(neighbor_flux_sum, i_idx, w * flux[j_idx])
    np.add.at(neighbor_flux_sum, j_idx, w * flux[i_idx])
    i_parent = flux + neighbor_flux_sum
    with np.errstate(divide='ignore', invalid='ignore'):
        blendedness = np.where(i_parent > 0, 1.0 - (flux / i_parent), 0.0)
    df["I_parent_nJy"] = i_parent
    df["blendedness"] = blendedness
    return df

print("=== STEP 1: rebuilding catalog with independent I_parent_nJy ===")
truth_full = pd.read_parquet(r"raw/truth_match/truth_tract3828.parquet")
truth_full = compute_blendedness_with_iparent(truth_full, ra_col="ra", dec_col="dec", flux_col="flux_r")

blend_lookup = truth_full[["id", "blendedness", "I_parent_nJy"]].drop_duplicates(subset="id")
blend_lookup = blend_lookup.rename(columns={"blendedness": "blendedness_truth"})

cleaned = pd.read_parquet(r"processed/cleaned_catalog.parquet")
cleaned = cleaned.drop(columns=["blendedness_truth", "I_parent_nJy"], errors="ignore")
cleaned = cleaned.merge(blend_lookup, on="id", how="left")

n_missing = cleaned["I_parent_nJy"].isna().sum()
print(f"Rows with no I_parent match: {n_missing:,} / {len(cleaned):,}")

cleaned.to_parquet(r"processed/cleaned_catalog_with_blendedness_truth.parquet", index=False)
print(f"Saved: processed/cleaned_catalog_with_blendedness_truth.parquet")
print(f"Columns confirmed present: {[c for c in cleaned.columns if c in ['blendedness_truth', 'I_parent_nJy', 'blendedness']]}")

=== STEP 1: rebuilding catalog with independent I_parent_nJy ===
Rows with no I_parent match: 0 / 132,830
Saved: processed/cleaned_catalog_with_blendedness_truth.parquet
Columns confirmed present: ['blendedness', 'blendedness_truth', 'I_parent_nJy']


In [5]:
# ============================================================
# STEP 2 — VAE evaluation, Kron aperture, leakage-free, linear-space
# ============================================================
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import sep
import tensorflow as tf
from pathlib import Path
from astropy.io import fits

from debvader.model.model import load_deblender
from debvader.normalize.normalize import normalize_non_linear

# ---- CONFIG ----
manifest_tf = pd.read_parquet(r"processed/tile_manifest_tf.parquet")
catalog = pd.read_parquet(r"processed/cleaned_catalog_with_blendedness_truth.parquet")

assert "I_parent_nJy" in catalog.columns, "I_parent_nJy missing — rerun Step 1 first"

BATCH_SIZE = 32
r_idx = 2
TARGET_SPLIT = "val"

out_dir = Path("processed/model_comparison_results")
out_dir.mkdir(parents=True, exist_ok=True)

catalog_indexed = catalog.set_index("id")


def to_normalized_space(raw_img):
    return normalize_non_linear(raw_img)


def aperture_flux_kron(img_2d, x, y, r_min=1.75, default_r=10.0):
    img = np.ascontiguousarray(img_2d.astype(np.float32))
    edge_mask = np.ones(img.shape, dtype=bool)
    edge_mask[10:-10, 10:-10] = False
    bkg_val = np.median(img[edge_mask]) if np.any(edge_mask) else 0.0
    img_sub = img - bkg_val
    try:
        kronrad, krflag = sep.kron_radius(img_sub, [x], [y], 3.0, 3.0, 0.0, 6.0)
        kr = kronrad[0]
        if not np.isfinite(kr) or kr <= 0:
            raise ValueError("invalid kron radius")
        r_use = max(kr * 2.5, r_min)
        flux, _, flag = sep.sum_ellipse(img_sub, [x], [y], 3.0, 3.0, 0.0, r_use, subpix=5)
        val = float(flux[0])
        return val if np.isfinite(val) else 0.0
    except Exception:
        flux, _, _ = sep.sum_circle(img_sub, [x], [y], default_r)
        val = float(flux[0])
        return val if np.isfinite(val) else 0.0


def get_photocalib_mean(fits_path):
    with fits.open(fits_path) as hdul:
        target_id = hdul[0].header["PHOTOCALIB_ID"]
        index = hdul[4].data
        match = index[index["id"] == target_id]
        cat_archive = match["cat.archive"][0]
        row0 = match["row0"][0]
        for hdu in hdul:
            if hdu.header.get("AR_CATN") == cat_archive:
                return float(hdu.data[row0]["calibrationMean"])
    return None

photocalib_cache = {}
def photocalib_for_patch(tier, patch_id, band="r"):
    key = (tier, patch_id)
    if key not in photocalib_cache:
        fpath = rf"RAW\IMAGES\{tier}\{band.upper()}\calexp-{band}-3828-{patch_id}.fits"
        photocalib_cache[key] = get_photocalib_mean(fpath)
    return photocalib_cache[key]


# ---- LOAD MODEL + SPLIT ----
vae_model = load_deblender(
    survey="dc2", input_shape=(59, 59, 6), latent_dim=32,
    filters=[32, 64, 128, 256], kernels=[3, 3, 3, 3],
)

subset = manifest_tf[manifest_tf["split"] == TARGET_SPLIT].reset_index(drop=True)
subset = subset[subset["object_id"].isin(catalog_indexed.index)].reset_index(drop=True)
print(f"Evaluating VAE (Kron aperture) on {len(subset)} tiles ({TARGET_SPLIT} split)")


# ---- RUN INFERENCE ----
records = []
for start in range(0, len(subset), BATCH_SIZE):
    batch_rows = subset.iloc[start:start + BATCH_SIZE]
    batch_arrays_raw = np.stack([np.load(p).astype(np.float32) for p in batch_rows["tile_path"]])
    batch_norm = to_normalized_space(batch_arrays_raw)

    reconstructed_dist = vae_model(tf.cast(batch_norm, tf.float32), training=False)
    recon_norm = reconstructed_dist.mean().numpy()

    for i, (_, row) in enumerate(batch_rows.iterrows()):
        obj_id = row["object_id"]
        I_parent = catalog_indexed.loc[obj_id, "I_parent_nJy"]
        blend_truth = catalog_indexed.loc[obj_id, "blendedness_truth"]
        blend_scarlet = catalog_indexed.loc[obj_id, "blendedness"]

        pc_mean = photocalib_for_patch(row["tier"], row["patch_id"])
        recon_r_raw = np.sinh(np.arctanh(np.clip(recon_norm[i][:, :, r_idx], -0.999999, 0.999999)))

        out_flux_counts = aperture_flux_kron(recon_r_raw, 29, 29)
        I_child_VAE_nJy = out_flux_counts * pc_mean

        if pd.notna(I_parent) and I_parent > 0 and np.isfinite(I_child_VAE_nJy):
            blendedness_VAE = np.clip(1 - (I_child_VAE_nJy / I_parent), 0, 1)
        else:
            blendedness_VAE = np.nan

        pixel_mse_norm = float(np.mean((batch_norm[i] - recon_norm[i]) ** 2))

        records.append({
            "object_id": obj_id, "patch_id": row["patch_id"], "tier": row["tier"],
            "blendedness_truth": blend_truth, "blendedness_SCARLET": blend_scarlet,
            "blendedness_VAE": blendedness_VAE, "I_child_VAE_nJy": I_child_VAE_nJy,
            "I_parent_nJy": I_parent, "pixel_mse_norm": pixel_mse_norm,
        })

    if (start // BATCH_SIZE) % 20 == 0 or (start + BATCH_SIZE) >= len(subset):
        print(f"Processed {min(start + BATCH_SIZE, len(subset))}/{len(subset)}")

df = pd.DataFrame(records)
df["blend_bin"] = pd.cut(df["blendedness_truth"], bins=[0, 0.02, 0.3, 1.0],
                          labels=["unblended", "blended", "severely_blended"])
df.to_parquet(out_dir / "VAE_results_kron.parquet", index=False)
print(f"\nSaved: {out_dir / 'VAE_results_kron.parquet'}")

df_valid = df.dropna(subset=["blendedness_VAE"]).copy()
print(f"Retained {len(df_valid)}/{len(df)} objects ({100*len(df_valid)/len(df):.1f}%)")
df_valid["abs_blend_error_VAE"] = (df_valid["blendedness_VAE"] - df_valid["blendedness_truth"]).abs()
df_valid["abs_blend_error_SCARLET"] = (df_valid["blendedness_SCARLET"] - df_valid["blendedness_truth"]).abs()

def bootstrap_median_ci(values, n_boot=2000, ci=95):
    values = values.dropna().values
    if len(values) < 10:
        return np.nan, np.nan, np.nan
    boot = [np.median(np.random.choice(values, size=len(values), replace=True)) for _ in range(n_boot)]
    lo, hi = np.percentile(boot, (100-ci)/2), np.percentile(boot, 100-(100-ci)/2)
    return np.median(values), lo, hi

print("\n=== PRIMARY (Kron): VAE blendedness error, median [95% CI] by bin ===")
for tier in ["unblended", "blended", "severely_blended"]:
    vals = df_valid[df_valid["blend_bin"] == tier]["abs_blend_error_VAE"]
    med, lo, hi = bootstrap_median_ci(vals)
    print(f"{tier:20s} median={med:.4f}  95% CI=[{lo:.4f}, {hi:.4f}]  n={len(vals)}")

print("\n=== COMPARISON: SCARLET blendedness error, median [95% CI] by bin ===")
for tier in ["unblended", "blended", "severely_blended"]:
    vals = df_valid[df_valid["blend_bin"] == tier]["abs_blend_error_SCARLET"]
    med, lo, hi = bootstrap_median_ci(vals)
    print(f"{tier:20s} median={med:.4f}  95% CI=[{lo:.4f}, {hi:.4f}]  n={len(vals)}")

print("\n=== SECONDARY: Pixel MSE (normalized space) by bin ===")
print(df.groupby("blend_bin")["pixel_mse_norm"].agg(
    median="median", p25=lambda x: x.quantile(0.25), p75=lambda x: x.quantile(0.75), count="count"))

in cropping
C:\Users\rohit\Downloads\GALAXY_DEBLENDING_DATASET\MODELS\debvader\src\debvader\data\weights\dc2
Evaluating VAE (Kron aperture) on 8524 tiles (val split)
Processed 32/8524
Processed 672/8524
Processed 1312/8524
Processed 1952/8524
Processed 2592/8524
Processed 3232/8524
Processed 3872/8524
Processed 5152/8524
Processed 5792/8524
Processed 6432/8524
Processed 7072/8524
Processed 7712/8524
Processed 8352/8524
Processed 8524/8524

Saved: processed\model_comparison_results\VAE_results_kron.parquet
Retained 8524/8524 objects (100.0%)

=== PRIMARY (Kron): VAE blendedness error, median [95% CI] by bin ===
unblended            median=0.0111  95% CI=[0.0105, 0.0116]  n=2531
blended              median=0.0746  95% CI=[0.0717, 0.0771]  n=5422
severely_blended     median=0.3850  95% CI=[0.3731, 0.4064]  n=571

=== COMPARISON: SCARLET blendedness error, median [95% CI] by bin ===
unblended            median=0.0087  95% CI=[0.0083, 0.0092]  n=2531
blended              median=0.0595  95% 

In [6]:
# ============================================================
# STEP 2 — VAE evaluation, Kron aperture, leakage-free, linear-space,
# NO background re-subtraction (confirmed tiles already sky-subtracted:
# border-pixel median/noise ratio = 0.033, re-subtracting was adding
# noise and risking contamination bias in blended tiles)
# ============================================================
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import sep
import tensorflow as tf
from pathlib import Path
from astropy.io import fits

from debvader.model.model import load_deblender
from debvader.normalize.normalize import normalize_non_linear

# ---- CONFIG ----
manifest_tf = pd.read_parquet(r"processed/tile_manifest_tf.parquet")
catalog = pd.read_parquet(r"processed/cleaned_catalog_with_blendedness_truth.parquet")

assert "I_parent_nJy" in catalog.columns, "I_parent_nJy missing — rerun Step 1 first"

BATCH_SIZE = 32
r_idx = 2
TARGET_SPLIT = "val"

out_dir = Path("processed/model_comparison_results")
out_dir.mkdir(parents=True, exist_ok=True)

catalog_indexed = catalog.set_index("id")


def to_normalized_space(raw_img):
    return normalize_non_linear(raw_img)


def aperture_flux_kron(img_2d, x, y, r_min=1.75, default_r=10.0):
    """No background subtraction — tiles are already sky-subtracted
    (confirmed empirically: border-pixel median is ~30x smaller than
    the noise level). Subtracting an estimated background here would
    only reintroduce noise and, in blended tiles, bias flux downward
    from neighbor contamination in the border region."""
    img = np.ascontiguousarray(img_2d.astype(np.float32))
    try:
        kronrad, krflag = sep.kron_radius(img, [x], [y], 3.0, 3.0, 0.0, 6.0)
        kr = kronrad[0]
        if not np.isfinite(kr) or kr <= 0:
            raise ValueError("invalid kron radius")
        r_use = max(kr * 2.5, r_min)
        flux, _, flag = sep.sum_ellipse(img, [x], [y], 3.0, 3.0, 0.0, r_use, subpix=5)
        val = float(flux[0])
        return val if np.isfinite(val) else 0.0
    except Exception:
        flux, _, _ = sep.sum_circle(img, [x], [y], default_r)
        val = float(flux[0])
        return val if np.isfinite(val) else 0.0


def get_photocalib_mean(fits_path):
    with fits.open(fits_path) as hdul:
        target_id = hdul[0].header["PHOTOCALIB_ID"]
        index = hdul[4].data
        match = index[index["id"] == target_id]
        cat_archive = match["cat.archive"][0]
        row0 = match["row0"][0]
        for hdu in hdul:
            if hdu.header.get("AR_CATN") == cat_archive:
                return float(hdu.data[row0]["calibrationMean"])
    return None

photocalib_cache = {}
def photocalib_for_patch(tier, patch_id, band="r"):
    key = (tier, patch_id)
    if key not in photocalib_cache:
        fpath = rf"RAW\IMAGES\{tier}\{band.upper()}\calexp-{band}-3828-{patch_id}.fits"
        photocalib_cache[key] = get_photocalib_mean(fpath)
    return photocalib_cache[key]


# ---- LOAD MODEL + SPLIT ----
vae_model = load_deblender(
    survey="dc2", input_shape=(59, 59, 6), latent_dim=32,
    filters=[32, 64, 128, 256], kernels=[3, 3, 3, 3],
)

subset = manifest_tf[manifest_tf["split"] == TARGET_SPLIT].reset_index(drop=True)
subset = subset[subset["object_id"].isin(catalog_indexed.index)].reset_index(drop=True)
print(f"Evaluating VAE (Kron aperture, no bkg re-subtraction) on {len(subset)} tiles ({TARGET_SPLIT} split)")


# ---- RUN INFERENCE ----
records = []
for start in range(0, len(subset), BATCH_SIZE):
    batch_rows = subset.iloc[start:start + BATCH_SIZE]
    batch_arrays_raw = np.stack([np.load(p).astype(np.float32) for p in batch_rows["tile_path"]])
    batch_norm = to_normalized_space(batch_arrays_raw)

    reconstructed_dist = vae_model(tf.cast(batch_norm, tf.float32), training=False)
    recon_norm = reconstructed_dist.mean().numpy()

    for i, (_, row) in enumerate(batch_rows.iterrows()):
        obj_id = row["object_id"]
        I_parent = catalog_indexed.loc[obj_id, "I_parent_nJy"]
        blend_truth = catalog_indexed.loc[obj_id, "blendedness_truth"]
        blend_scarlet = catalog_indexed.loc[obj_id, "blendedness"]

        pc_mean = photocalib_for_patch(row["tier"], row["patch_id"])
        recon_r_raw = np.sinh(np.arctanh(np.clip(recon_norm[i][:, :, r_idx], -0.999999, 0.999999)))

        out_flux_counts = aperture_flux_kron(recon_r_raw, 29, 29)
        I_child_VAE_nJy = out_flux_counts * pc_mean

        if pd.notna(I_parent) and I_parent > 0 and np.isfinite(I_child_VAE_nJy):
            blendedness_VAE = np.clip(1 - (I_child_VAE_nJy / I_parent), 0, 1)
        else:
            blendedness_VAE = np.nan

        pixel_mse_norm = float(np.mean((batch_norm[i] - recon_norm[i]) ** 2))

        records.append({
            "object_id": obj_id, "patch_id": row["patch_id"], "tier": row["tier"],
            "blendedness_truth": blend_truth, "blendedness_SCARLET": blend_scarlet,
            "blendedness_VAE": blendedness_VAE, "I_child_VAE_nJy": I_child_VAE_nJy,
            "I_parent_nJy": I_parent, "pixel_mse_norm": pixel_mse_norm,
        })

    if (start // BATCH_SIZE) % 20 == 0 or (start + BATCH_SIZE) >= len(subset):
        print(f"Processed {min(start + BATCH_SIZE, len(subset))}/{len(subset)}")

df = pd.DataFrame(records)
df["blend_bin"] = pd.cut(df["blendedness_truth"], bins=[0, 0.02, 0.3, 1.0],
                          labels=["unblended", "blended", "severely_blended"])
df.to_parquet(out_dir / "VAE_results_kron_nobkg.parquet", index=False)
print(f"\nSaved: {out_dir / 'VAE_results_kron_nobkg.parquet'}")

df_valid = df.dropna(subset=["blendedness_VAE"]).copy()
print(f"Retained {len(df_valid)}/{len(df)} objects ({100*len(df_valid)/len(df):.1f}%)")
df_valid["abs_blend_error_VAE"] = (df_valid["blendedness_VAE"] - df_valid["blendedness_truth"]).abs()
df_valid["abs_blend_error_SCARLET"] = (df_valid["blendedness_SCARLET"] - df_valid["blendedness_truth"]).abs()

def bootstrap_median_ci(values, n_boot=2000, ci=95):
    values = values.dropna().values
    if len(values) < 10:
        return np.nan, np.nan, np.nan
    boot = [np.median(np.random.choice(values, size=len(values), replace=True)) for _ in range(n_boot)]
    lo, hi = np.percentile(boot, (100-ci)/2), np.percentile(boot, 100-(100-ci)/2)
    return np.median(values), lo, hi

print("\n=== PRIMARY (Kron, no bkg re-subtraction): VAE blendedness error, median [95% CI] by bin ===")
for tier in ["unblended", "blended", "severely_blended"]:
    vals = df_valid[df_valid["blend_bin"] == tier]["abs_blend_error_VAE"]
    med, lo, hi = bootstrap_median_ci(vals)
    print(f"{tier:20s} median={med:.4f}  95% CI=[{lo:.4f}, {hi:.4f}]  n={len(vals)}")

print("\n=== COMPARISON: SCARLET blendedness error, median [95% CI] by bin ===")
for tier in ["unblended", "blended", "severely_blended"]:
    vals = df_valid[df_valid["blend_bin"] == tier]["abs_blend_error_SCARLET"]
    med, lo, hi = bootstrap_median_ci(vals)
    print(f"{tier:20s} median={med:.4f}  95% CI=[{lo:.4f}, {hi:.4f}]  n={len(vals)}")

print("\n=== SECONDARY: Pixel MSE (normalized space) by bin ===")
print(df.groupby("blend_bin")["pixel_mse_norm"].agg(
    median="median", p25=lambda x: x.quantile(0.25), p75=lambda x: x.quantile(0.75), count="count"))

in cropping
C:\Users\rohit\Downloads\GALAXY_DEBLENDING_DATASET\MODELS\debvader\src\debvader\data\weights\dc2
Evaluating VAE (Kron aperture, no bkg re-subtraction) on 8524 tiles (val split)
Processed 32/8524
Processed 672/8524
Processed 1312/8524
Processed 1952/8524
Processed 2592/8524
Processed 3232/8524
Processed 3872/8524
Processed 4512/8524
Processed 5152/8524
Processed 5792/8524
Processed 6432/8524
Processed 7072/8524
Processed 7712/8524
Processed 8352/8524
Processed 8524/8524

Saved: processed\model_comparison_results\VAE_results_kron_nobkg.parquet
Retained 8524/8524 objects (100.0%)

=== PRIMARY (Kron, no bkg re-subtraction): VAE blendedness error, median [95% CI] by bin ===
unblended            median=0.0112  95% CI=[0.0106, 0.0117]  n=2531
blended              median=0.0747  95% CI=[0.0720, 0.0771]  n=5422
severely_blended     median=0.3870  95% CI=[0.3757, 0.4040]  n=571

=== COMPARISON: SCARLET blendedness error, median [95% CI] by bin ===
unblended            median=0.0087  

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import tensorflow as tf
import sep

from debvader.model.model import load_deblender
from debvader.normalize.normalize import normalize_non_linear

# Paths
df_path = Path("processed/model_comparison_results/VAE_results_kron_nobkg.parquet")
manifest_path = Path("processed/tile_manifest_tf.parquet")
out_dir = Path("processed/figures")
out_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(df_path).dropna(subset=["blendedness_VAE", "blendedness_truth"])
manifest = pd.read_parquet(manifest_path).set_index("object_id")

# Load VAE model
vae_model = load_deblender(
    survey="dc2", input_shape=(59, 59, 6), latent_dim=32,
    filters=[32, 64, 128, 256], kernels=[3, 3, 3, 3],
)

def asinh_stretch(img, a=0.05):
    """Applies arcsinh stretch for better dynamic range visualization."""
    return np.arcsinh(img / a)

# ============================================================
# NEW FIGURE 5 — Reconstruction Inspection: Input, VAE, Residual
# ============================================================
sample_ids = []
for tier in ["unblended", "blended", "severely_blended"]:
    sub = df[df["blend_bin"] == tier]
    # Grab 1 median error case per tier
    sub["err"] = (sub["blendedness_VAE"] - sub["blendedness_truth"]).abs()
    med_case = sub.sort_values("err").iloc[len(sub)//2]
    sample_ids.append((tier, med_case["object_id"]))

fig, axes = plt.subplots(len(sample_ids), 3, figsize=(9, 3 * len(sample_ids)))
r_idx = 2

for row_idx, (tier, obj_id) in enumerate(sample_ids):
    tile_path = manifest.loc[obj_id, "tile_path"]
    raw_tile = np.load(tile_path).astype(np.float32)
    norm_tile = normalize_non_linear(np.expand_dims(raw_tile, axis=0))
    
    # Run VAE inference
    recon_norm = vae_model(tf.cast(norm_tile, tf.float32), training=False).mean().numpy()[0]
    
    # Unscale r-band
    raw_r = raw_tile[:, :, r_idx]
    recon_r = np.sinh(np.arctanh(np.clip(recon_norm[:, :, r_idx], -0.99, 0.99)))
    residual = np.abs(raw_r - recon_r)
    
    # Plot Raw Input
    ax_in = axes[row_idx, 0]
    ax_in.imshow(asinh_stretch(raw_r), cmap="inferno", origin="lower")
    ax_in.set_title(f"Input ({tier})\nObj ID: {obj_id}", fontsize=9)
    ax_in.axis("off")
    
    # Plot VAE Reconstruction
    ax_rec = axes[row_idx, 1]
    ax_rec.imshow(asinh_stretch(recon_r), cmap="inferno", origin="lower")
    ax_rec.set_title(f"VAE Reconstruction\nBlend Err: {df.loc[df['object_id']==obj_id, 'blendedness_VAE'].values[0] - df.loc[df['object_id']==obj_id, 'blendedness_truth'].values[0]:.3f}", fontsize=9)
    ax_rec.axis("off")
    
    # Plot Residual Map
    ax_res = axes[row_idx, 2]
    im_res = ax_res.imshow(asinh_stretch(residual), cmap="magma", origin="lower")
    ax_res.set_title("Absolute Residual\n|Input - VAE|", fontsize=9)
    ax_res.axis("off")
    plt.colorbar(im_res, ax=ax_res, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(out_dir / "fig5_reconstruction_residuals.png", dpi=200)
plt.close()
print("Saved upgraded fig5_reconstruction_residuals.png")


# ============================================================
# NEW FIGURE 6 — Radial Surface Brightness Profiles
# ============================================================
def get_radial_profile(img, cx=29, cy=29, max_r=25):
    y, x = np.indices(img.shape)
    r = np.sqrt((x - cx)**2 + (y - cy)**2)
    r_int = r.astype(int)
    radial_mean = [np.mean(img[r_int == i]) for i in range(max_r)]
    return np.arange(max_r), radial_mean

fig, ax = plt.subplots(figsize=(7, 4.5))

for tier, color in zip(["unblended", "blended", "severely_blended"], ["#55A868", "#C44E52", "#8172B2"]):
    sub = df[df["blend_bin"] == tier]
    sample_obj = sub.iloc[0]["object_id"]
    tile_path = manifest.loc[sample_obj, "tile_path"]
    
    raw_tile = np.load(tile_path).astype(np.float32)
    norm_tile = normalize_non_linear(np.expand_dims(raw_tile, axis=0))
    recon_norm = vae_model(tf.cast(norm_tile, tf.float32), training=False).mean().numpy()[0]
    
    raw_r = raw_tile[:, :, r_idx]
    recon_r = np.sinh(np.arctanh(np.clip(recon_norm[:, :, r_idx], -0.99, 0.99)))
    
    radii, raw_prof = get_radial_profile(raw_r)
    _, recon_prof = get_radial_profile(recon_r)
    
    ax.plot(radii, raw_prof, label=f"Raw ({tier})", linestyle="--", color=color, alpha=0.7)
    ax.plot(radii, recon_prof, label=f"VAE ({tier})", linestyle="-", color=color, lw=2)

ax.set_yscale("log")
ax.set_xlabel("Radius from Center (pixels)")
ax.set_ylabel("Mean Pixel Intensity (Log Scale)")
ax.set_title("1D Radial Surface Brightness Profile Recovery")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(out_dir / "fig6_radial_profiles.png", dpi=150)
plt.close()
print("Saved fig6_radial_profiles.png")

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import tensorflow as tf

from debvader.model.model import load_deblender
from debvader.normalize.normalize import normalize_non_linear

# ---- PATHS & SETUP ----
df_path = Path("processed/model_comparison_results/VAE_results_kron_nobkg.parquet")
manifest_path = Path("processed/tile_manifest_tf.parquet")
out_dir = Path("processed/figures/galleries")
out_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(df_path).dropna(subset=["blendedness_VAE", "blendedness_truth"])
manifest = pd.read_parquet(manifest_path).set_index("object_id")

# Absolute error metric
df["abs_err"] = (df["blendedness_VAE"] - df["blendedness_truth"]).abs()
df["raw_err"] = df["blendedness_VAE"] - df["blendedness_truth"]

# Load pre-trained VAE
vae_model = load_deblender(
    survey="dc2", input_shape=(59, 59, 6), latent_dim=32,
    filters=[32, 64, 128, 256], kernels=[3, 3, 3, 3],
)

def asinh_stretch(img, a=0.05):
    """Applies arcsinh scaling for low-surface-brightness visibility."""
    return np.arcsinh(img / a)

r_idx = 2  # r-band index

# ---- GENERATE GALLERIES FOR EACH TIER AND EXTREME ----
tiers = ["unblended", "blended", "severely_blended"]

for tier in tiers:
    sub = df[df["blend_bin"] == tier].copy()
    
    # Extract top 16 best and worst
    top16_best = sub.nsmallest(16, "abs_err")
    top16_worst = sub.nlargest(16, "abs_err")
    
    for category, sample_df in [("Best", top16_best), ("Worst", top16_worst)]:
        obj_ids = sample_df["object_id"].tolist()
        
        # 1. Batch Load & Process Tiles
        tile_paths = [manifest.loc[oid, "tile_path"] for oid in obj_ids]
        raw_tiles = np.stack([np.load(p).astype(np.float32) for p in tile_paths])
        norm_tiles = normalize_non_linear(raw_tiles)
        
        # 2. Batch VAE Inference
        recon_norms = vae_model(tf.cast(norm_tiles, tf.float32), training=False).mean().numpy()
        
        # 3. Unscale r-band
        raw_r = raw_tiles[:, :, :, r_idx]
        recon_r = np.sinh(np.arctanh(np.clip(recon_norms[:, :, :, r_idx], -0.99, 0.99)))
        residuals = np.abs(raw_r - recon_r)
        
        # 4. Plot 8x6 Grid (8 rows, 2 objects per row)
        fig, axes = plt.subplots(8, 6, figsize=(14, 18))
        fig.suptitle(f"Top 16 {category} Cases — Tier: {tier.upper()}", fontsize=14, y=0.995, fontweight="bold")
        
        for i in range(16):
            r = i // 2       # Row index (0 to 7)
            c_off = (i % 2) * 3  # Column offset (0 or 3)
            
            oid = obj_ids[i]
            row_meta = sample_df.iloc[i]
            err_val = row_meta["raw_err"]
            truth_val = row_meta["blendedness_truth"]
            vae_val = row_meta["blendedness_VAE"]
            
            # Input Stamp
            ax_in = axes[r, c_off]
            ax_in.imshow(asinh_stretch(raw_r[i]), cmap="inferno", origin="lower")
            ax_in.set_title(f"Truth B={truth_val:.2f}", fontsize=7)
            ax_in.axis("off")
            
            # VAE Recon
            ax_rec = axes[r, c_off + 1]
            ax_rec.imshow(asinh_stretch(recon_r[i]), cmap="inferno", origin="lower")
            ax_rec.set_title(f"VAE B={vae_val:.2f}", fontsize=7)
            ax_rec.axis("off")
            
            # Residual
            ax_res = axes[r, c_off + 2]
            ax_res.imshow(asinh_stretch(residuals[i]), cmap="magma", origin="lower")
            ax_res.set_title(f"Err={err_val:+.3f}", fontsize=7, color="crimson" if category=="Worst" else "green")
            ax_res.axis("off")
        
        plt.tight_layout()
        save_path = out_dir / f"gallery_{tier}_{category.lower()}16.png"
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        plt.close()
        print(f"Saved: {save_path}")